In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from openai import OpenAI

import utils.prompt_manager as prompt_manager
import utils.pdf_to_text as pdf_to_text
import utils.create_sanitized_folder as create_sanitized_folder
import utils.copy_pdf_to_folder as copy_pdf_to_folder
import model.model as model

# Your Open AI api
client = OpenAI()

# Paths
pdf_path = r"dataset/csb_reports/BethlehemFinal.pdf"
incident_cards_path = "runs/incident_cards"
hazards_json = "prompt\hazards.json"
conditions_json = "prompt\conditions.json"

report_text = pdf_to_text.pdf_to_text(pdf_path)
# identify_name, identify_incident, simplify_incident, identify_deviations_equipments, identify_hazard, 
# identify_condition, relate_hazards, relate_scenario, chain_events, chain_scenario, finalize_scenario
all_prompts = prompt_manager.prompts.load_all()
model_name = "gpt-5"

# identify the csb report's name through pdf file
identify_name_output = model.run_prompt(
    prompt=all_prompts["identify_name"],
    variables={"report_text": report_text[:2000]},
    output_dir=None,
    model_name=model_name,
    prompt_key="identify_name"
)

# create folder
folder = create_sanitized_folder.create_sanitized_folder(identify_name_output, content=identify_name_output, base_dir=incident_cards_path)
copied_pdf = copy_pdf_to_folder.copy_pdf_to_folder(pdf_path, folder)

identify_incident_output = model.run_prompt(
    prompt=all_prompts["identify_incident"],
    variables={"report_text": report_text},
    output_dir=folder,
    model_name=model_name,
    prompt_key="identify_incident"
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P2' is a

'You are a professional process safety analyst.\n\nREPORT_TEXT: {report_text}\n\nTASK:\n- Identify the name of the report from REPORT_TEXT\n\nExample:\n- T2_Laboratories_Inc_Runaway_Reaction_Four_Killed_32_Injured\n- Hayes_Lemmerz_International_Huntington_Aluminum_Dust_Explosion_1_Killed_6_Injured\n- Chemical_Reaction_Hydrogen_Release_Explosion_and_Fire_at_AB_Specialty_Silicones'

In [6]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import csv
import re
import pdfplumber
from openai import OpenAI
import utils.prompt_manager as prompt_manager
import model.model as model

# === 基础设置 ===
client = OpenAI()
incident_cards_path = Path("runs/incident_cards")
model_name = "gpt-5"

# === 加载所有prompt ===
all_prompts = prompt_manager.prompts.load_all()

def extract_first_2000_chars(pdf_path: Path) -> str:
    """使用 pdfplumber 提取前2000个字符"""
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
            if len(text) >= 2000:
                break
    return text[:2000]

def sanitize_name(name: str) -> str:
    """将模型输出转换为安全文件夹名：去除非法字符并用_替换空格和标点"""
    # 去掉首尾空白
    name = name.strip()
    # 替换所有非字母数字字符（包括空格、标点）为 "_"
    name = re.sub(r"[^0-9A-Za-z]+", "_", name)
    # 去掉重复的连续下划线
    name = re.sub(r"_+", "_", name)
    # 去掉开头和结尾的下划线
    name = name.strip("_")
    return name

def rename_folder_based_on_identified_name(folder: Path, summary_records: list):
    """从PDF提取内容并根据Identify Name重命名文件夹"""
    pdf_files = list(folder.glob("*.pdf"))
    if not pdf_files:
        print(f"⚠️ No PDF found in {folder}")
        summary_records.append([folder.name, "", "No PDF found"])
        return
    
    pdf_path = pdf_files[0]
    print(f"\n📄 Processing: {pdf_path.name}")

    # 提取前2000字符
    report_excerpt = extract_first_2000_chars(pdf_path)
    
    # 调用identify_name prompt
    try:
        identify_name_output = model.run_prompt(
            prompt=all_prompts["identify_name"],
            variables={"report_text": report_excerpt},
            output_dir=None,
            model_name=model_name,
            prompt_key="identify_name"
        ).strip()
    except Exception as e:
        print(f"❌ Error identifying name for {folder.name}: {e}")
        summary_records.append([folder.name, "", f"Error: {e}"])
        return

    # 生成安全的新文件夹名
    safe_name = sanitize_name(identify_name_output)
    if not safe_name:
        print(f"⚠️ Empty or invalid name for {folder.name}, skipping rename.")
        summary_records.append([folder.name, "", "Empty or invalid name"])
        return

    new_folder_path = folder.parent / safe_name
    if new_folder_path.exists():
        print(f"⚠️ Folder {new_folder_path.name} already exists, skipping rename.")
        summary_records.append([folder.name, new_folder_path.name, "Already exists"])
        return
    
    # 执行重命名
    folder.rename(new_folder_path)
    print(f"✅ Renamed: {folder.name} → {new_folder_path.name}")
    summary_records.append([folder.name, new_folder_path.name, identify_name_output])

# === 主程序 ===
summary_records = [["Old Folder Name", "New Folder Name", "Identified Report Name"]]

for subfolder in incident_cards_path.iterdir():
    if subfolder.is_dir():
        rename_folder_based_on_identified_name(subfolder, summary_records)

# === 导出CSV ===
summary_file = incident_cards_path / "summary.csv"
with open(summary_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerows(summary_records)

print(f"\n🎯 All folders processed. Summary saved to: {summary_file}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
⚠️ No PDF found in runs\incident_cards\68

📄 Processing: 20160412_Macondo_Full_Exec_Summary.pdf

🚀 Running prompt: identify_name
⏳ Generating...

Drilling_Rig_Explosion_and_Fire_at_the_Macondo_Well

✅ Done!

⚙️ Skipped saving (output_dir=None)

✅ Renamed: 65 → Drilling_Rig_Explosion_and_Fire_at_the_Macondo_Well

📄 Processing: ab_specialty_investigation_report_final21.pdf

🚀 Running prompt: identify_name
⏳ Generating...

Chemical_Reaction_Hydrogen_Release_Explosion_and_Fire_at_AB_Specialty_Silicones

✅ Done!

⚙️ Skipped saving (output_dir=None)

✅ Renamed: 18 → Chemical_Reaction_Hydrogen_Release_Explosion_and_Fire_at_AB_Specialty_Silicones

📄 Processing: Aghorn_Investigation_Report_-_Board_Approved_Version_(5-4-21)_.pdf

🚀 Running prompt: identify_name
⏳ Generating...

Hydrogen_Sulfide_Release_at_Aghorn_Operating_Waterflood_Station

✅ Done!

⚙️ Skipped saving (output_dir=None)

✅ Renamed: 53 → Hydrog

In [2]:
from pathlib import Path
import csv

# === 基础路径 ===
base_path = Path("runs/incident_cards")

# === 获取所有子文件夹并排序 ===
subfolders = sorted([f for f in base_path.iterdir() if f.is_dir()], key=lambda x: x.name.lower())

# === 输出日志文件 ===
log_file = base_path / "rename_log.csv"
records = [["Old Folder Name", "New Folder Name"]]

# === 遍历重命名 ===
for i, folder in enumerate(subfolders, start=1):
    new_name = str(i)
    new_path = folder.parent / new_name

    # 避免重名（如果已有该编号，就加个后缀）
    if new_path.exists():
        new_path = folder.parent / f"{i}_dup"

    folder.rename(new_path)
    print(f"✅ Renamed: {folder.name} → {new_path.name}")
    records.append([folder.name, new_path.name])

# === 写入CSV日志 ===
with open(log_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerows(records)

print(f"\n🎯 All folders renamed successfully. Log saved to: {log_file}")


✅ Renamed: 2025_FEEDBACK_FORM → 1
✅ Renamed: adfa → 2
✅ Renamed: adfAFA → 3
✅ Renamed: Allied_Terminals_Inc_Catastrophic_Tank_Collapse_CSB_Investigation_Report → 4
✅ Renamed: asdas → 5
✅ Renamed: asdasd → 6
✅ Renamed: asdasdsad → 7
✅ Renamed: asdsa → 8
✅ Renamed: asfs → 9
✅ Renamed: Barton_Solvents_Static_Spark_Ignites_Explosion_Inside_Flammable_Liquid_Storage_Tank_No_2007_06_I_KS → 10
✅ Renamed: Cabin_Creek_Hydroelectric_Plant_Final_Investigation_Report → 11
✅ Renamed: Carbide_Industries_Case_Study_Board_Voting_Copy_February_2013_ → 12
✅ Renamed: Case_Study_Hot_Work_Control_and_Safe_Work_Practices_at_Oil_and_Gas_Production_Wells → 13
✅ Renamed: Case_Study_Mixing_and_Heating_a_Flammable_Liquid_in_an_Open_Top_Tank → 14
✅ Renamed: Catastrophic_Vessel_Failure_D_D_Williamson_Co_Inc_Louisville_Kentucky_April_11_2003 → 15
✅ Renamed: Chemical_Reaction_Decomposition_and_Toxic_Gas_Release_at_Bio-Lab_Inc → 16
✅ Renamed: Chemical_Reaction_Decomposition_and_Toxic_Gas_Release_at_Bio_Lab_Inc → 17
✅ 